# 09 — Live TI tracker (run daily during the tournament)

One place to refresh: pull the live books, recompute the derivative fair
values from tournament-to-date statistics, and re-check the winner market
against the model. Everything below reads the manual state block first —
update those numbers from the day's results, then run all cells.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
# ---- LIVE STATE (update daily) ----
STATE = dict(
    games_played   = 0,      # total GAMES (not series) completed at TI
    radiant_wins   = 0,
    games_left     = 180,
    longest_so_far = 0.0,    # minutes
    prior_p        = 0.530,  # current-patch radiant rate (notebook 07)
)
TI_BAN_COUNTS = {}           # hero_id -> bans so far at TI
STATE

In [ ]:
from src.polymarket import all_ti_books
from src.derivatives import radiant_dire_market, longest_game_market, compare_to_book
books = all_ti_books()
w = books['winner']
print(f"WINNER BOOK | vol ${w.attrs['volume']:,.0f} | 24h ${w.attrs['volume24h']:,.0f} | "
      f"priced-sum {w.attrs['priced_sum']:.3f}")
w[['outcome','mid','bid','ask','spread','volume24h']].head(16)

In [ ]:
r = radiant_dire_market(STATE['games_played'], STATE['radiant_wins'],
                        STATE['games_left'], STATE['prior_p'], 400)
rd = books['radiant_dire']
print('model:', r['fair_prices'])
print(rd[['outcome','mid','bid','ask']].to_string(index=False))
model = pd.DataFrame({'outcome': list(r['fair_prices']), 'p': list(r['fair_prices'].values())})
compare_to_book(model, dict(zip(rd.outcome, rd.mid)), 'outcome', 'p')

In [ ]:
# Rolling check: is THIS tournament's radiant rate drifting from the prior?
if STATE['games_played'] >= 20:
    obs = STATE['radiant_wins'] / STATE['games_played']
    se  = (obs * (1 - obs) / STATE['games_played']) ** 0.5
    print(f"TI-to-date radiant rate {obs:.3f} +- {se:.3f} (95% CI "
          f"{obs-1.96*se:.3f}-{obs+1.96*se:.3f}) vs prior {STATE['prior_p']:.3f}")
    print('If the CI excludes the prior, trust the tournament sample more (raise its weight).')
else:
    print('too few games for a meaningful in-tournament read')

### Daily checklist
1. Update `STATE` and `TI_BAN_COUNTS` from the day's results.
2. Re-run notebooks 06/07 if new parsed matches were collected.
3. Re-price all three derivative books; act only on edges that survive
   costs **and** sit in a book with no competing bid.
4. Cancel resting orders before a team's elimination series (adverse
   selection), and re-check after every patch/roster event.